# Beauty Try-On Benchmark — فاز صفر / نسخهٔ 0.1.0

این نوت‌بوک دو موتور را اجرا می‌کند: انتقال مدل و رنگ مو با **HairFastGAN**، و رنگ مو با **SegFormer-B0 + LAB**. مدل‌های پایه متعلق به سازندگان اصلی‌اند؛ سهم این پروژه اجرای Colab، ارزیابی، سنجه‌ها، کنترل پردازش تصویر و بنچمارک است.

**روش اجرا:** Runtime → Change runtime type → T4 GPU → Save، سپس Run all.
۲۳ سلول شماره‌دار و خوانا داریم. سلول‌های `%%writefile` فقط کد می‌نویسند؛ فایل عکس مشتری ایجاد نمی‌کنند. نصب در ۴، وزن‌ها در ۶، سلامت در ۱۶، دموها در ۱۷ و ۱۸ و گزارش در ۱۹ است. **سلول ۲۰ به‌طور پیش‌فرض روشن است و عکس فرد و مرجع را از تو می‌گیرد.** اگر مهلت انتخاب عکس تمام شد، فقط سلول ۲۰ را دوباره اجرا کن.

تست اختیاری رنگ روی عکس تو در ۲۱ و بنچمارک ZIP در ۲۲ هستند و پیش‌فرض خاموش‌اند. پس از اجرای اولیه، برای عکس تازه فقط ۲۰ را اجرا کن؛ نصب را تکرار نکن. اگر Colab محیط اجرای جدیدی ساخته باشد، کد نوت‌بوک و خروجی‌های قبلی فایل‌های نصب را بازنمی‌گردانند و باید مراحل آماده‌سازی دوباره اجرا شوند.

**حریم خصوصی:** دریافت با FileReader در حافظه، پاکسازی EXIF/GPS، عدم استخراج ZIP و عدم ذخیرهٔ عکس خصوصی. نمایش عکس خصوصی با RPC موقت و Blob مرورگر است؛ بعد از ۵ دقیقه، بستن پیش‌نمایش یا نمایش نتیجهٔ بعدی پاک می‌شود. نمایش به خروجی استاندارد تصویری نوت‌بوک ارسال نمی‌شود. بااین‌حال پاکسازی امن همهٔ نسخه‌های پنهان Python/CUDA/مرورگر و نگهداری داده توسط Google قابل تضمین یا ممیزی مستقل این کد نیست. قاب دموهای عمومی ممکن است در خروجی ذخیره‌شدهٔ نوت‌بوک بماند.

**وضعیت:** یک نمونهٔ حرفه‌ای جداگانه در Colab در ۵ اکتبر ۲۰۲۶ روی Tesla T4 با موفقیت اجرا و سنجه‌های تک‌دمو ثبت شده است؛ این موضوع فقط مدرک feasibility است و به معنی GPU-validation بایت‌به‌بایت این revision نیست. آزمون کامل ۳۰ عکس و گیت کیفیت هنوز تکمیل نشده‌اند. این نوت‌بوک محصول نهایی سالن، اپ موبایل یا پنل کاربری نیست. محدودیت‌های SEAN و مجوزهای مدل‌ها را پیش از استفادهٔ تجاری بررسی کنید.

**کپی‌رایت:** کد و مستندات اصلی این پروژه © 2026 Fereshteh Abolghasemi و تحت MIT منتشر می‌شوند؛ مدل‌ها و وابستگی‌های شخص ثالث متعلق به صاحبان خودشان هستند.


## سلول 1 — بررسی GPU و آماده‌سازی مسیر

شروع یک اجرای جدید؛ هیچ عکس خصوصی دریافت نمی‌شود.


In [ ]:
import os, pathlib, shutil, subprocess, sys
_root = pathlib.Path('/content/beauty_phase0')
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
_code.mkdir(parents=True, exist_ok=True)
(_code.parent/'tests').mkdir(parents=True, exist_ok=True)
for name in ('_beauty_benchmark_api', '_beauty_professional_v2_api', '_beauty_professional_readable_api'):
    previous = sys.modules.get(name)
    if previous is not None:
        previous.stop_engine()
    sys.modules.pop(name, None)
for name in ('runtime', 'controller', 'photo_preview'):
    sys.modules.pop(name, None)
os.environ['BEAUTY_ROOT'] = str(_root)
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
(_code/'run_status.json').write_text('{}')
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Runtime → Change runtime type → T4 GPU را انتخاب کن.')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'], check=True)
if shutil.disk_usage(_root).free < 18 * 2**30 and not (_root/'weight_manifest.json').exists():
    raise RuntimeError('نصب اول حداقل ۱۸ GiB فضای آزاد لازم دارد.')
print('GPU بررسی شد؛ مراحل بعد کد و وزن‌های عمومی را آماده می‌کنند.')


## سلول 2 — تنظیمات و مدیریت اجرا

مسیرها، مدیریت مدل و وضعیت مراحل؛ فقط نوشتن کد.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/runtime.py
"""Notebook orchestration. Image requests remain in controller/worker memory."""
import contextlib, hashlib, importlib.util, json, os, pathlib, subprocess, sys, time
from collections import deque

ROOT = pathlib.Path(os.environ.get('BEAUTY_ROOT', '/content/beauty_phase0')).resolve()
CODE_ROOT = pathlib.Path(__file__).resolve().parent
TEST_ROOT = pathlib.Path(os.environ.get('BEAUTY_TEST_ROOT', str(CODE_ROOT.parent / 'tests')))
REPO = ROOT / 'HairFastGAN'
VENV = ROOT / 'venv'
PYTHON = str(VENV / 'bin/python')
CODE_REV = '49e98019c9243d1991c19d45764fc3340a498de2'
WEIGHTS_REV = '47855f8ea72b02294649385211a1ae1fd7dcb58e'
SEG_MODEL = 'isjackwild/segformer-b0-finetuned-segments-skin-hair-clothing'
SEG_REV = 'dd02ee5b73017377ba6f12a58c3e300cbbfd58c8'
OP_BACKEND = os.environ.get('BEAUTY_OP_BACKEND', 'auto')
ENV = dict(os.environ, BEAUTY_ROOT=str(ROOT), BEAUTY_OP_BACKEND=OP_BACKEND,
           BEAUTY_TEST_DEMO=str(ROOT/'public_demo'),
           WANDB_MODE='disabled', HF_HUB_DISABLE_TELEMETRY='1',
           HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
           PYTHONDONTWRITEBYTECODE='1', MAX_JOBS='2',
           TORCH_CUDA_ARCH_LIST='7.5;8.9')

def run(*command, env=None, timeout=1800):
    """Stream public setup/test output; include its actual error in the exception."""
    command = list(map(str, command))
    tail = deque(maxlen=60)
    child_env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONDONTWRITEBYTECODE='1',
                     CMAKE_BUILD_PARALLEL_LEVEL='2', MAX_JOBS='2')
    child_env['PATH'] = str(VENV / 'bin') + os.pathsep + child_env.get('PATH', '')
    if env:
        child_env.update(env)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=child_env) as child:
        # A watchdog bounds stalled setup even when no new output arrives.
        import threading
        expired = threading.Event()
        def terminate():
            expired.set()
            child.kill()
        timer = threading.Timer(timeout, terminate)
        timer.daemon = True
        timer.start()
        try:
            for line in child.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            code = child.wait()
        finally:
            timer.cancel()
    if expired.is_set():
        raise TimeoutError('Setup/test timeout:\n' + ''.join(tail))
    if code:
        raise RuntimeError('Command failed (exit %s):\n%s' % (code, ''.join(tail)))

def write_json(path, value):
    path = pathlib.Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2))
    temporary.replace(path)

@contextlib.contextmanager
def stage(number, title):
    """Persist anonymous stage status; never photos, masks, input names or latents."""
    path = CODE_ROOT / 'run_status.json'
    state = json.loads(path.read_text()) if path.exists() else {}
    item = {'status':'RUNNING', 'title':title, 'started_at':time.time()}
    state[str(number)] = item
    write_json(path, state)
    print('\n[%s] %s — شروع' % (number, title), flush=True)
    began = time.perf_counter()
    try:
        yield
    except Exception as exc:
        item.update(status='FAILED', seconds=time.perf_counter()-began,
                    error_type=type(exc).__name__)
        write_json(path, state)
        print('[%s] متوقف شد؛ علت خطا در خروجی همین سلول است.' % number, flush=True)
        raise
    else:
        item.update(status='PASSED', seconds=time.perf_counter()-began)
        write_json(path, state)
        print('[%s] کامل شد (%.1f ثانیه)' % (number, item['seconds']), flush=True)

def preflight():
    import shutil
    if not shutil.which('nvidia-smi'):
        raise RuntimeError('Runtime → Change runtime type → T4 GPU را انتخاب کن.')
    run('nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv')
    if shutil.disk_usage(ROOT).free < 18 * 2**30 and not (ROOT/'weight_manifest.json').exists():
        raise RuntimeError('برای نصب اول حداقل ۱۸ GiB فضای آزاد لازم است.')
    print('Python مدل مستقل از Python فعلی Colab ساخته می‌شود.', flush=True)

def _script(name):
    run(sys.executable, str(CODE_ROOT/name))

def install():
    _script('install.py')

def assets():
    _script('assets.py')

def verify():
    if not pathlib.Path(PYTHON).is_file():
        raise RuntimeError('سلول ۴ نصب را اجرا کن؛ محیط مدل موجود نیست.')
    probe = '''
import torch, torchvision, numpy, cv2, dlib, clip, transformers, pkg_resources
assert torch.cuda.is_available(), "CUDA unavailable"
x=torch.ones((16,16),device="cuda"); y=x@x; torch.cuda.synchronize()
assert y[0,0].item()==16
print("CUDA arithmetic passed:",torch.cuda.get_device_name())
print("Torch:",torch.__version__,"NumPy:",numpy.__version__,"OpenCV:",cv2.__version__)
'''
    run(PYTHON, '-c', probe)
    run(PYTHON, str(TEST_ROOT/'test_core.py'), env=ENV)
    run(PYTHON, str(TEST_ROOT/'test_runtime.py'), env=ENV)
    run(PYTHON, str(TEST_ROOT/'test_operators.py'),
        env=dict(ENV, BEAUTY_TEST_REPO=str(REPO)))

def api():
    name = '_beauty_benchmark_api'
    if name not in sys.modules:
        spec = importlib.util.spec_from_file_location(name, CODE_ROOT/'controller.py')
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        spec.loader.exec_module(module)
    module = sys.modules[name]
    module.configure(ROOT, CODE_ROOT, PYTHON, ENV)
    return module

def demo(service, show=True):
    if service not in {'hairstyle','color'}:
        raise ValueError('Unknown service')
    module = api()
    face = reference = response = None
    previous = module.PRIVATE_IMAGES
    try:
        module.start_engine(service)
        module.PRIVATE_IMAGES = False  # Official PUBLIC images only.
        face = bytearray((ROOT/'public_demo/6.png').read_bytes())
        reference = bytearray((ROOT/'public_demo/7.png').read_bytes())
        response = module.evaluate(face, reference, show=show, retain_output=not show, repeats=3)
        metrics = response if show else response['metrics']
        if metrics['status'] != 'ok':
            print(json.dumps(metrics, ensure_ascii=False, indent=2), flush=True)
            raise RuntimeError(metrics.get('public_diagnostic') or metrics['error_code'])
        write_json(CODE_ROOT/(service+'_demo_metrics.json'), metrics)
        if not show:
            destination = ROOT/'public_demo_outputs'
            destination.mkdir(parents=True, exist_ok=True)
            (destination/(service+'_result.png')).write_bytes(response['output_png'])
            (destination/(service+'_comparison.png')).write_bytes(response['comparison_png'])
            print('Public demo comparison:', destination/(service+'_comparison.png'))
        print('مدل:',service, '| زمان خالص:',round(metrics['inference_s'],3),'s',
              '| زمان پایپ‌لاین:',round(metrics['pipeline_s'],3),'s',
              '| Peak VRAM:',round(metrics['peak_allocated_mib'],1),'MiB',flush=True)
        return metrics
    finally:
        module.clear_buffer(face)
        module.clear_buffer(reference)
        if response is not None:
            module.clear_buffer(response.get('output_png'))
            module.clear_buffer(response.get('comparison_png'))
        module.PRIVATE_IMAGES = previous
        module.stop_engine()

def summary():
    from IPython.display import display, Markdown
    rows = []
    incomplete = []
    state_path=CODE_ROOT/'run_status.json'
    state=json.loads(state_path.read_text()) if state_path.exists() else {}
    for stage_number, service, limit in [(17,'hairstyle',8.0),(18,'color',0.5)]:
        path = CODE_ROOT/(service+'_demo_metrics.json')
        if not path.is_file() or state.get(str(stage_number),{}).get('status')!='PASSED':
            rows.append('|%s|اجرا نشده|—|—|—|—|' % service)
            incomplete.append(str(stage_number))
            continue
        data = json.loads(path.read_text())
        rows.append('|%s|%s|%.3f|%.3f|%.1f|%s|' % (
            service,data['status'],data['inference_s'],data['pipeline_s'],data['peak_allocated_mib'],
            'زیر هدف زمان دمو' if data['inference_s'] < limit else 'بالاتر از هدف زمان دمو'))
    display(Markdown('| موتور | وضعیت | زمان خالص (s) | پایپ‌لاین (s) | Peak VRAM (MiB) | زمان دمو |\n'
                     '|---|---|---:|---:|---:|---|\n'+'\n'.join(rows)))
    print('این جدول نتیجهٔ دموی عمومی است؛ گیت ۳۰ عکس هنوز ارزیابی نشده است.',flush=True)
    if incomplete:
        raise RuntimeError('دموی کامل تأیید نشد؛ سلول‌های ' + ' و '.join(incomplete)
                           + ' هنوز در همین اجرا موفق نشده‌اند.')

def sample(service='hairstyle', public=False):
    module=api(); previous=module.PRIVATE_IMAGES
    try:
        module.PRIVATE_IMAGES=not public
        module.start_engine(service)
        module.single_demo(show=public)
    finally:
        module.PRIVATE_IMAGES=previous
        module.stop_engine()

def batch(service='hairstyle', public=False, folder=None, save_outputs=False, review=True):
    module=api(); previous=module.PRIVATE_IMAGES
    try:
        module.PRIVATE_IMAGES=not public
        module.start_engine(service)
        preview_fn = None
        if review and not public:
            from photo_preview import show_transient
            preview_fn = show_transient
        options=dict(show=public,score_fn=module.record_scores if public or review else None,
                     preview_fn=preview_fn,
                     save_dir=str(ROOT/'public_benchmark_outputs') if save_outputs else None,
                     allow_public_export=public and save_outputs)
        records=module.batch_folder(folder,**options) if folder else module.batch_zip_demo(**options)
        gate=module.quality_gate(records,seconds=8.0 if service=='hairstyle' else .5)
        print(json.dumps(gate,ensure_ascii=False,indent=2),flush=True)
        module.export_metrics(records,path=str(CODE_ROOT/(service+'_benchmark.csv')),
                              seconds=8.0 if service=='hairstyle' else .5)
        return gate
    finally:
        module.PRIVATE_IMAGES=previous
        module.stop_engine()

def stop():
    api().stop_engine()


## سلول 3 — کد نصب مستقل

Python 3.10، نسخه‌های ثابت، CLIP و dlib؛ نصب در سلول بعد است.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/install.py
import sys, os, json, hashlib, shutil
from runtime import ROOT, REPO, VENV, PYTHON, CODE_ROOT, run
ROOT.mkdir(parents=True,exist_ok=True)
run(sys.executable,'-m','pip','install','uv==0.6.17')
if not (VENV/'bin/python').is_file():
    run(sys.executable,'-m','uv','venv','--python','3.10',str(VENV))
PACKAGES = ['setuptools==69.5.1', 'wheel==0.43.0', 'packaging==24.2', 'numpy==1.26.4', 'Pillow==10.4.0', 'scipy==1.11.4', 'scikit-image==0.22.0', 'scikit-learn==1.4.2', 'matplotlib==3.8.4', 'pandas==2.2.2', 'PyYAML==6.0.2', 'addict==2.4.0', 'dill==0.3.8', 'face_alignment==1.3.4', 'huggingface-hub==0.26.5', 'transformers==4.44.2', 'safetensors==0.4.5', 'ninja==1.11.1.3', 'nvidia-ml-py==12.560.30', 'ftfy==6.2.3', 'gdown==5.2.0', 'torchmetrics==0.11.4', 'tqdm==4.66.5', 'requests==2.32.3', 'tensorboardX==2.6.2.2', 'tensorboard==2.17.1', 'clip @ git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33', 'opencv-python==4.10.0.84', 'numba==0.59.1', 'llvmlite==0.42.0', 'protobuf==4.25.3']
run(sys.executable,'-m','uv','pip','install','--python',PYTHON,
    'torch==2.2.2','torchvision==0.17.2',
    '--index-url','https://download.pytorch.org/whl/cu121')
if not shutil.which('g++'):
    run('apt-get','update','-qq')
    run('apt-get','install','-y','-qq','build-essential')
# Install the legacy build backend and a known CMake in the model environment.
run(sys.executable,'-m','uv','pip','install','--python',PYTHON,
    'setuptools==69.5.1','wheel==0.43.0','packaging==24.2','cmake==3.29.6')
run(sys.executable,'-m','uv','pip','install','--python',PYTHON,
    '--no-build-isolation-package','clip',*PACKAGES)
# Separating dlib makes compilation progress/failures visible in their own block.
print('Building/validating dlib 19.24.6 with bounded compilation jobs',flush=True)
run(sys.executable,'-m','uv','pip','install','--python',PYTHON,'dlib==19.24.6')
run(PYTHON,'-c','import clip, pkg_resources, cv2, dlib, torch; print("CLIP and dlib imports passed")')
import subprocess
freeze=subprocess.run([sys.executable,'-m','uv','pip','freeze','--python',PYTHON],
                      check=True,stdout=subprocess.PIPE,text=True).stdout
(CODE_ROOT/'requirements.resolved.txt').write_text(freeze)
print('Resolved dependency versions recorded.',flush=True)


## سلول 4 — نصب کتابخانه‌ها

نصب اول ممکن است برای کامپایل dlib زمان‌بر باشد؛ منتظر پایان همین سلول بمان.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

with ctx.stage(4, 'نصب کتابخانه‌ها'):
    ctx.install()


## سلول 5 — کد دریافت وزن‌ها و اصلاح اپراتورها

نسخه‌های ثابت و fallback ثبت‌شونده؛ هیچ وزن یا عکس خصوصی داخل خود نوت‌بوک نیست.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/assets.py
import ast, json, pathlib, sys, os, shutil
from runtime import ROOT, REPO, VENV, PYTHON, CODE_ROOT, CODE_REV, WEIGHTS_REV, SEG_MODEL, SEG_REV, OP_BACKEND, run
if not (REPO/'.git').is_dir():
    run('git','clone','https://github.com/AIRI-Institute/HairFastGAN.git',REPO)
run('git','-C',REPO,'fetch','origin',CODE_REV)
run('git','-C',REPO,'checkout','--force',CODE_REV)
patches = []
root_upfirdn = (REPO / 'models/stylegan2/op/upfirdn2d.py').read_text()
native_upfirdn = root_upfirdn[root_upfirdn.index('\ndef upfirdn2d_native('):]
for path in REPO.rglob('op/*.py'):
    if path.name not in {'fused_act.py', 'upfirdn2d.py'}:
        continue
    source = path.read_text()
    if path.name == 'upfirdn2d.py' and 'from torch.nn import functional as F' not in source:
        # The FSE copy has a native implementation but omitted its functional import.
        source = 'from torch.nn import functional as F\n' + source
    tree = ast.parse(source)
    node = next((n for n in tree.body if isinstance(n, ast.Assign)
                 and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Name)
                 and n.value.func.id == 'load'), None)
    if node is None:
        raise RuntimeError('Upstream operator structure changed: ' + str(path.relative_to(REPO)))
    variable = node.targets[0].id
    lines = source.splitlines(keepends=True)
    original = ''.join(lines[node.lineno - 1:node.end_lineno])
    guarded = ('try:\n'
               "    if os.environ.get('BEAUTY_OP_BACKEND') == 'torch_reference':\n"
               "        raise RuntimeError('reference_backend_requested')\n"
               + ''.join('    ' + line for line in original.splitlines(keepends=True))
               + '\nexcept Exception:\n    ' + variable + ' = None\n')
    source = ''.join(lines[:node.lineno - 1]) + guarded + ''.join(lines[node.end_lineno:])
    if path.name == 'fused_act.py':
        start = source.index('\ndef fused_leaky_relu(')
        source = source[:start] + '''
def fused_leaky_relu(input, bias, negative_slope=0.2, scale=2 ** 0.5):
    if fused is not None and input.device.type == 'cuda':
        return FusedLeakyReLUFunction.apply(input, bias, negative_slope, scale)
    rest = [1] * (input.ndim - bias.ndim - 1)
    return torch.nn.functional.leaky_relu(input + bias.view(1, bias.shape[0], *rest), negative_slope) * scale
'''
    else:
        start = source.index('\ndef upfirdn2d(')
        end = source.index('\ndef upfirdn2d_native(', start)
        source = source[:start] + '''
def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):
    if upfirdn2d_op is not None and input.device.type == 'cuda':
        return UpFirDn2d.apply(input, kernel, (up, up), (down, down), (pad[0], pad[1], pad[0], pad[1]))
    return upfirdn2d_native(input, kernel, up, up, down, down, pad[0], pad[1], pad[0], pad[1])

''' + native_upfirdn
    ast.parse(source)
    path.write_text(source)
    patches.append(str(path.relative_to(REPO)))
# Fix removed Pillow alias in official alignment code; no resize-geometry change.
for path in REPO.rglob('*.py'):
    source = path.read_text()
    if 'PIL.Image.ANTIALIAS' in source:
        path.write_text(source.replace('PIL.Image.ANTIALIAS', 'PIL.Image.Resampling.LANCZOS'))


WEIGHTS = [
    'ArcFace/backbone_ir50.pth', 'BiSeNet/face_parsing_79999_iter.pth',
    'Blending/checkpoint.pth', 'FeatureStyleEncoder/143_enc.pth',
    'FeatureStyleEncoder/79999_iter.pth', 'FeatureStyleEncoder/backbone.pth',
    'FeatureStyleEncoder/psp_ffhq_encode.pt', 'PostProcess/latent_avg.pt',
    'PostProcess/pp_model.pth', 'Rotate/rotate_best.pth', 'ShapeAdaptor/mask_generator.pth',
    'ShapeAdaptor/shape_predictor_68_face_landmarks.dat', 'StyleGAN/ffhq.pt',
    'StyleGAN/ffhq_PCA.npz', 'encoder4editing/e4e_ffhq_encode.pt',
    'sean_checkpoints/CelebA-HQ_pretrained/latest_net_G.pth',
]
download_code = '''
from huggingface_hub import snapshot_download
import pathlib, json, hashlib, clip
repo, revision, seg_model, seg_revision, root, files = json.loads(__import__('sys').argv[1])
snapshot_download(repo_id='AIRI-Institute/HairFastGAN', revision=revision,
                  allow_patterns=['pretrained_models/' + f for f in files], local_dir=repo,
                  max_workers=2)
snapshot_download(repo_id='AIRI-Institute/HairFastGAN', revision=revision,
                  allow_patterns=['input/6.png','input/7.png'], local_dir=repo, max_workers=2)
demo = pathlib.Path(root) / 'public_demo'
demo.mkdir(exist_ok=True)
for name in ('6.png','7.png'):
    __import__('shutil').copyfile(pathlib.Path(repo)/'input'/name, demo/name)
snapshot_download(repo_id=seg_model, revision=seg_revision,
                  allow_patterns=['config.json','preprocessor_config.json','model.safetensors'],
                  local_dir=root + '/segformer', max_workers=2)
# Force public CLIP weight download before the strict offline request guard is armed.
clip.load('ViT-B/32', device='cpu', download_root=root + '/clip')
manifest = {}
for file in files:
    p = pathlib.Path(repo) / 'pretrained_models' / file
    if not p.is_file() or p.stat().st_size < 1024:
        raise RuntimeError('Missing/invalid model asset: ' + file)
    with p.open('rb') as stream:
        h = hashlib.sha256()
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    manifest[file] = {'bytes': p.stat().st_size, 'sha256': h.hexdigest()}
(pathlib.Path(root) / 'weight_manifest.json').write_text(json.dumps(manifest, indent=2))
'''
run(PYTHON, '-c', download_code, json.dumps([str(REPO), WEIGHTS_REV, SEG_MODEL, SEG_REV, str(ROOT), WEIGHTS]))
(ROOT / 'run_config.json').write_text(json.dumps({
    'code_revision': CODE_REV, 'weights_revision': WEIGHTS_REV,
    'segformer_model': SEG_MODEL, 'segformer_revision': SEG_REV,
    'operator_backend_requested': OP_BACKEND, 'patches': patches,
    'python': '3.10', 'torch': '2.2.2+cu121', 'precision': 'float32',
}, indent=2))

config=json.loads((ROOT/'run_config.json').read_text())
config['notebook_version']='beauty_tryon_benchmark_0.1.0'
config['resolved_requirements_sha256']=__import__('hashlib').sha256((CODE_ROOT/'requirements.resolved.txt').read_bytes()).hexdigest()
(ROOT/'run_config.json').write_text(json.dumps(config,indent=2))
cfg=json.loads((ROOT/'segformer/config.json').read_text())
assert [int(k) for k,v in cfg['id2label'].items() if v.lower()=='hair']==[2]
assert len(patches)==6, 'All six StyleGAN operator copies must be patched'
print('16 HairFastGAN assets, segmentation configuration and six operators prepared.',flush=True)


## سلول 6 — دریافت وزن‌های عمومی

۱۶ وزن HairFastGAN، سگمنتیشن و نمونه‌های عمومی گرم‌سازی دریافت می‌شوند.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

with ctx.stage(6, 'دریافت وزن‌ها'):
    ctx.assets()


## سلول 7 — بافر تصویر، EXIF و LAB

اصلاح بسته‌شدن بافر PNG از ابتدا اعمال شده است.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/core.py
import io
import math
import warnings
import numpy as np
import cv2
from PIL import Image, ImageOps

MAX_FILE_BYTES = 25 * 1024 * 1024
MAX_PIXELS = 16_000_000
MAX_SIDE = 1600


class WipingBytesIO(io.BytesIO):
    """Zero the owned encoded-image buffer before any caller closes it."""
    def close(self):
        if self.closed:
            return
        view = self.getbuffer()
        try:
            view[:] = b'\0' * len(view)
        finally:
            view.release()
        super().close()


def wipe(value):
    """Best effort only: Python/PIL/CUDA can make immutable or hidden copies."""
    if isinstance(value, bytearray):
        value[:] = b'\0' * len(value)
    elif isinstance(value, np.ndarray) and value.flags.writeable:
        value.fill(0)
    elif isinstance(value, Image.Image):
        try:
            # Inspect Pillow's stored core directly instead of the public `im`
            # property. Newer Pillow versions can raise when `im` is read after
            # close; touching a lazy rejected image can also trigger decoding.
            core = value.__dict__.get('_im')
            is_live_core = core is not None and core.__class__.__name__ != 'DeferredError'
            if is_live_core:
                try:
                    value.paste(0, (0, 0, value.width, value.height))
                except (ValueError, AssertionError):
                    # Pillow may already have destroyed the pixel core.
                    pass
        finally:
            try:
                value.close()
            except (ValueError, AssertionError):
                pass
    elif isinstance(value, io.BytesIO):
        if value.closed:
            return
        view = value.getbuffer()
        try:
            view[:] = b'\0' * len(view)
        finally:
            view.release()
            value.close()


def decode_clean(data):
    """Apply EXIF orientation, then create a fresh RGB image with no metadata."""
    if not isinstance(data, (bytes, bytearray)) or not 0 < len(data) <= MAX_FILE_BYTES:
        raise ValueError('invalid_image_bytes')
    stream = WipingBytesIO(data)
    raw = oriented = rgb = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('error', Image.DecompressionBombWarning)
            raw = Image.open(stream)
            if raw.format not in {'JPEG', 'PNG', 'WEBP'}:
                raise ValueError('unsupported_image_format')
            if raw.width * raw.height > MAX_PIXELS or getattr(raw, 'n_frames', 1) != 1:
                raise ValueError('image_limits_exceeded')
            raw.load()
            oriented = ImageOps.exif_transpose(raw)
            rgb = oriented.convert('RGB')
            rgb.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)
            clean = Image.new('RGB', rgb.size)
            clean.paste(rgb)
            clean.info.clear()
            return clean
    finally:
        # The image is fully loaded before this point; erase encoded bytes first.
        # PNG Image.close() can close caller-owned streams, so close is idempotent.
        try:
            wipe(stream)
        finally:
            for image in (rgb, oriented, raw):
                if image is not None:
                    wipe(image)


def encode_png(image):
    """No EXIF, GPS, ICC, text chunks, filenames or source metadata."""
    fresh = Image.new('RGB', image.size)
    fresh.paste(image)
    stream = io.BytesIO()
    try:
        fresh.save(stream, format='PNG', compress_level=3)
        return bytearray(stream.getvalue())
    finally:
        wipe(fresh)
        wipe(stream)


def lab_recolor(rgb, mask, target_rgb, strength=0.85):
    """Keep L* exactly in LAB before sRGB conversion; change chroma only.
    Dark-to-blond lightening is deliberately outside this baseline.
    Clipping/quantization can slightly change measured output luminance.
    Pixels outside the nonzero mask are bit-for-bit identical.
    """
    if rgb.dtype != np.uint8 or rgb.ndim != 3 or rgb.shape[2] != 3:
        raise ValueError('invalid_rgb')
    if mask.shape != rgb.shape[:2] or not np.isfinite(mask).all():
        raise ValueError('invalid_mask')
    if not 0 <= strength <= 1:
        raise ValueError('invalid_strength')
    target = np.asarray(target_rgb, dtype=np.float32)
    if target.shape != (3,) or not np.isfinite(target).all() or (target < 0).any() or (target > 255).any():
        raise ValueError('invalid_target_color')
    lab = changed = rendered = delta = None
    try:
        lab = cv2.cvtColor(rgb.astype(np.float32) / 255, cv2.COLOR_RGB2LAB)
        target_lab = cv2.cvtColor((target / 255).reshape(1, 1, 3), cv2.COLOR_RGB2LAB)[0, 0]
        selection = mask >= 0.75
        if selection.sum() < 32:
            raise ValueError('insufficient_hair_pixels')
        source_ab = np.median(lab[selection, 1:3], axis=0)
        delta = target_lab[1:3] - source_ab
        changed = lab.copy()
        changed[..., 1:3] += delta * (np.clip(mask, 0, 1) * strength)[..., None]
        rendered = np.rint(np.clip(cv2.cvtColor(changed, cv2.COLOR_LAB2RGB), 0, 1) * 255).astype(np.uint8)
        rendered[mask <= 0] = rgb[mask <= 0]
        result = rendered.copy()
        return result
    finally:
        for x in (lab, changed, rendered, delta):
            if x is not None:
                wipe(x)


def comparison_frame(original, aligned, reference, final):
    # Original and aligned are both shown: alignment must never hide identity changes.
    panels = [('Original', original), ('Aligned target', aligned),
              ('Reference', reference), ('Result (aligned crop)', final)]
    from PIL import ImageDraw
    frame = Image.new('RGB', (4 * 384, 416), 'white')
    draw = ImageDraw.Draw(frame)
    for i, (title, image) in enumerate(panels):
        thumb = ImageOps.contain(image, (384, 384), Image.Resampling.LANCZOS)
        frame.paste(thumb, (i * 384 + (384 - thumb.width) // 2, 32 + (384 - thumb.height) // 2))
        draw.text((i * 384 + 8, 9), title, fill='black')
        wipe(thumb)
    return frame


## سلول 8 — موتور انتقال مدل مو

چهره و رفرنس باید هرکدام دقیقاً یک چهرهٔ قابل تشخیص داشته باشند.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/hair_engine.py
import sys, time
from core import np, wipe

class HairEngine:
    def __init__(self, root):
        import torch, dlib, clip
        import functools
        # All downloaded public weights remain on disk. Client images never do.
        original_load = clip.load
        clip.load = functools.partial(original_load, download_root=str(root / 'clip'))
        from hair_swap import HairFast, get_parser
        from utils.shape_predictor import align_face
        args = get_parser().parse_args([])
        args.save_all = False
        args.batch_size = 1  # Sequential embeddings lower the T4 memory requirement.
        self.model = HairFast(args)
        # CLIP's official loader chooses FP16 on CUDA by default; normalize this
        # benchmark profile explicitly rather than mislabeling mixed precision.
        for module in (self.model.net, self.model.embed, self.model.align, self.model.blend):
            module.float().eval()
        self.predictor = dlib.shape_predictor('pretrained_models/ShapeAdaptor/shape_predictor_68_face_landmarks.dat')
        self.detector = dlib.get_frontal_face_detector()
        self.align_face = align_face
        self.backend = {}
        for name, module in list(sys.modules.items()):
            if name.endswith('.op.fused_act') or name.endswith('.op.upfirdn2d'):
                variable = 'fused' if name.endswith('fused_act') else 'upfirdn2d_op'
                self.backend[name] = 'cuda_fused' if getattr(module, variable, None) is not None else 'torch_reference'
        # Disable any optional SEAN latent-code exports.
        for sub in self.model.align.sean_model.modules():
            if hasattr(sub, 'save_npy'):
                sub.save_npy = False

    def clear_private_state(self):
        # Upstream Embedding retains its last dataset/dataloader even after return.
        embed = self.model.embed
        data = getattr(embed, 'dataset', None)
        if data is not None:
            import torch
            for image in getattr(data, 'images', []):
                if isinstance(image, torch.Tensor):
                    image.zero_()
            data.images = []
            data.names = []
        for attr in ('dataset', 'dataloader'):
            if hasattr(embed, attr):
                delattr(embed, attr)

    def run(self, face, reference):
        import torch
        import torchvision.transforms.functional as TF
        aligned = []
        returned = None
        try:
            # Reject ambiguous multi-face inputs rather than silently choosing a face.
            for image in (face, reference):
                array = np.array(image)
                try:
                    if len(self.detector(array, 1)) != 1:
                        raise ValueError('exactly_one_face_required')
                finally:
                    wipe(array)
            t0 = time.perf_counter()
            aligned = self.align_face([face, reference], predictor=self.predictor, return_tensors=True)
            torch.cuda.synchronize()
            alignment_s = time.perf_counter() - t0
            t1 = time.perf_counter()
            returned = self.model.swap(aligned[0], aligned[1], aligned[1], align=False,
                                       benchmark=False, seed=3407)
            torch.cuda.synchronize()
            inference_s = time.perf_counter() - t1
            result = TF.to_pil_image(returned.detach().cpu())
            aligned_face = TF.to_pil_image(aligned[0])
            aligned_ref = TF.to_pil_image(aligned[1])
            return result, aligned_face, aligned_ref, {'alignment_s': alignment_s, 'inference_s': inference_s}
        finally:
            if returned is not None:
                returned.zero_()
            for tensor in aligned:
                tensor.zero_()
            self.clear_private_state()


## سلول 9 — موتور رنگ مستقل

سگمنتیشن و تغییر a*/b* در LAB؛ روشن‌کردن شدید موی تیره پوشش داده نمی‌شود.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/color_engine.py
import json, time
from core import np, cv2, Image, wipe, lab_recolor

class ColorEngine:
    def __init__(self, root, side=512):
        import torch
        from transformers import SegformerForSemanticSegmentation
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            str(root / 'segformer'), local_files_only=True, use_safetensors=True).eval().cuda()
        self.side = side
        config = json.loads((root / 'segformer/preprocessor_config.json').read_text())
        self.mean = torch.tensor(config['image_mean'], device='cuda').view(1, 3, 1, 1)
        self.std = torch.tensor(config['image_std'], device='cuda').view(1, 3, 1, 1)
        labels = self.model.config.id2label
        hits = [int(key) for key, value in labels.items() if value.lower() == 'hair']
        if len(hits) != 1:
            raise RuntimeError('hair_label_missing')
        self.hair_id = hits[0]
        self.backend = {'segformer': 'torch_fp32', 'input_size': side, 'hair_label': self.hair_id}

    def mask(self, rgb):
        import torch
        import torch.nn.functional as F
        small = cv2.resize(rgb, (self.side, self.side), interpolation=cv2.INTER_LINEAR)
        x = torch.from_numpy(small).permute(2, 0, 1).unsqueeze(0).cuda().float() / 255
        logits = probs = hair = hard = confidence = None
        array = hard_array = None
        try:
            logits = self.model(pixel_values=(x - self.mean) / self.std).logits
            probs = F.interpolate(logits, size=rgb.shape[:2], mode='bilinear', align_corners=False).softmax(1)
            hair = probs[:, self.hair_id]
            # Never dilate hair onto face. Soften only inside the hard hair support.
            hard = probs.argmax(1) == self.hair_id
            confidence = ((hair - .35) / .5).clamp(0, 1) * hard
            array = confidence[0].cpu().numpy().copy()
            hard_array = hard[0].cpu().numpy()
            soft = cv2.GaussianBlur(array, (3, 3), 0.7)
            soft *= hard_array
            soft[soft < .02] = 0
            wipe(array)
            return soft
        finally:
            wipe(small)
            for a in (array, hard_array):
                if a is not None:
                    wipe(a)
            for t in (x, logits, probs, hair, hard, confidence):
                if t is not None:
                    t.zero_()

    def clear_private_state(self):
        pass

    def run(self, face, reference=None, target_rgb=(116, 52, 40), strength=.85, reference_color=False):
        rgb = np.array(face)
        mask = ref_rgb = ref_mask = out = None
        t0 = time.perf_counter()
        try:
            mask = self.mask(rgb)
            if reference_color:
                if reference is None:
                    raise ValueError('color_reference_required')
                ref_rgb = np.array(reference)
                ref_mask = self.mask(ref_rgb)
                selected = ref_mask >= .75
                if selected.sum() < 32:
                    raise ValueError('insufficient_reference_hair')
                # Approximate median RGB swatch; lighting affects this measurement.
                target_rgb = np.median(ref_rgb[selected], axis=0).tolist()
            out = lab_recolor(rgb, mask, target_rgb, strength)
            import torch
            torch.cuda.synchronize()
            inference_s = time.perf_counter() - t0
            return Image.fromarray(out.copy()), face.copy(), (reference or face).copy(), {
                'alignment_s': 0., 'inference_s': inference_s,
                'color_target_rgb': target_rgb, 'hair_fraction': float((mask > 0).mean()),
                'reference_color': reference_color,
            }
        finally:
            for x in (rgb, mask, ref_rgb, ref_mask, out):
                if x is not None:
                    wipe(x)


## سلول 10 — پردازشگر، زمان و VRAM

Warmup عمومی، IPC حافظه‌ای و محافظ نوشتن/شبکه در درخواست.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/worker_tail.py
from hair_engine import HairEngine
from color_engine import ColorEngine
import os, sys, gc, time, json, struct, pickle, threading, contextlib
from pathlib import Path

GUARD_ACTIVE = False


def audit_guard(event, args):
    if not GUARD_ACTIVE:
        return
    if event == 'open':
        mode = args[1] if len(args) > 1 else None
        flags = args[2] if len(args) > 2 else 0
        write_flags = os.O_WRONLY | os.O_RDWR | os.O_CREAT | os.O_TRUNC | os.O_APPEND
        if (isinstance(mode, str) and any(c in mode for c in 'wax+')) or (isinstance(flags, int) and flags & write_flags):
            raise PermissionError('disk_write_blocked_during_request')
    if event in {'os.remove', 'os.rename', 'os.mkdir', 'os.rmdir', 'os.link', 'os.symlink',
                 'subprocess.Popen', 'os.system', 'socket.connect', 'socket.bind', 'socket.getaddrinfo'}:
        raise PermissionError('side_effect_blocked_during_request')


@contextlib.contextmanager
def request_guard():
    global GUARD_ACTIVE
    before = GUARD_ACTIVE
    GUARD_ACTIVE = True
    try:
        yield
    finally:
        GUARD_ACTIVE = before


class MemorySampler:
    """NVML samples process/device usage; PyTorch reports exact allocator peaks.
    NVML samples may miss spikes <10 ms; allocator peaks do not.
    """
    def __init__(self):
        self.process_peak = None
        self.device_peak = None
        self.stop_event = threading.Event()
        try:
            import pynvml
            self.nvml = pynvml
            pynvml.nvmlInit()
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        except Exception:
            self.nvml = None

    def sample(self):
        if self.nvml is None:
            return
        try:
            total = self.nvml.nvmlDeviceGetMemoryInfo(self.handle).used / 2**20
            self.device_peak = max(self.device_peak or 0, total)
            procs = self.nvml.nvmlDeviceGetComputeRunningProcesses(self.handle)
            used = sum(p.usedGpuMemory for p in procs if p.pid == os.getpid()
                       and isinstance(p.usedGpuMemory, int) and p.usedGpuMemory < 2**60)
            if used:
                self.process_peak = max(self.process_peak or 0, used / 2**20)
        except Exception:
            pass

    def start(self):
        self.sample()
        def loop():
            while not self.stop_event.wait(.01):
                self.sample()
        self.thread = threading.Thread(target=loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.sample()
        self.stop_event.set()
        self.thread.join(timeout=1)






def evaluate(engine, request, provenance):
    import torch
    face = reference = result = aligned = aligned_ref = frame = None
    measurements = []
    output = {}
    started = time.perf_counter()
    try:
        face = decode_clean(request['face'])
        if request.get('reference'):
            reference = decode_clean(request['reference'])
        if isinstance(engine, HairEngine) and reference is None:
            raise ValueError('hairstyle_reference_required')
        decode_s = time.perf_counter() - started
        repetitions = int(request.get('repeats', 3))
        if not 1 <= repetitions <= 10:
            raise ValueError('invalid_repeats')
        for _ in range(repetitions):
            for image in (result, aligned, aligned_ref):
                if image is not None:
                    wipe(image)
            result = aligned = aligned_ref = None
            sampler = MemorySampler()
            sampler.start()
            torch.cuda.synchronize()
            baseline_alloc = torch.cuda.memory_allocated()
            torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            try:
                with torch.inference_mode():
                    if isinstance(engine, ColorEngine):
                        result, aligned, aligned_ref, m = engine.run(
                            face, reference, request.get('target_rgb', (116, 52, 40)),
                            float(request.get('strength', .85)), bool(request.get('reference_color', False)))
                    else:
                        result, aligned, aligned_ref, m = engine.run(face, reference)
                torch.cuda.synchronize()
                # Includes CPU detection/alignment and output tensor->PIL conversion.
                m['pipeline_s'] = time.perf_counter() - t0
                m['peak_allocated_mib'] = torch.cuda.max_memory_allocated() / 2**20
                m['peak_reserved_mib'] = torch.cuda.max_memory_reserved() / 2**20
                m['incremental_allocated_mib'] = (torch.cuda.max_memory_allocated() - baseline_alloc) / 2**20
            finally:
                sampler.stop()
            m['nvml_process_peak_mib'] = sampler.process_peak
            m['nvml_device_peak_mib'] = sampler.device_peak
            measurements.append(m)
        # Images leave the worker only when deliberately requested for public/synthetic QA.
        if request.get('return_images', False):
            output['output_png'] = encode_png(result)
            frame = comparison_frame(face, aligned, aligned_ref, result)
            output['comparison_png'] = encode_png(frame)
        last = measurements[-1]
        output['metrics'] = {
            'status': 'ok', 'error_code': None,
            'inference_s': float(np.median([m['inference_s'] for m in measurements])),
            'pipeline_s': float(np.median([m['pipeline_s'] for m in measurements])),
            'alignment_s': float(np.median([m['alignment_s'] for m in measurements])),
            'decode_s': decode_s,
            'peak_allocated_mib': max(m['peak_allocated_mib'] for m in measurements),
            'peak_reserved_mib': max(m['peak_reserved_mib'] for m in measurements),
            'nvml_process_peak_mib': max((m['nvml_process_peak_mib'] for m in measurements if m['nvml_process_peak_mib'] is not None), default=None),
            'nvml_device_peak_mib': max((m['nvml_device_peak_mib'] for m in measurements if m['nvml_device_peak_mib'] is not None), default=None),
            'repeats': repetitions, 'measurements': measurements,
            'input_size': list(face.size), 'output_size': list(result.size),
            'alignment': 'ffhq_1024_crop' if isinstance(engine, HairEngine) else 'none',
            'backend': engine.backend, 'provenance': provenance,
        }
        if isinstance(engine, ColorEngine):
            output['metrics']['color_target_rgb'] = last['color_target_rgb']
            output['metrics']['hair_fraction'] = last['hair_fraction']
        return output
    finally:
        for image in (face, reference, result, aligned, aligned_ref, frame):
            if image is not None:
                wipe(image)
        engine.clear_private_state()
        # Request-owned mutable buffers are zeroed even on decode/OOM/model failure.
        for name in ('face', 'reference'):
            wipe(request.get(name))
            request.pop(name, None)
        gc.collect()
        torch.cuda.empty_cache()  # Releases unused blocks, not a secure erasure guarantee.


def read_exact(stream, n):
    value = bytearray(n)
    view = memoryview(value)
    pos = 0
    while pos < n:
        count = stream.readinto(view[pos:])
        if not count:
            wipe(value)
            raise EOFError('worker_pipe_closed')
        pos += count
    view.release()
    return value


def read_message(stream):
    header = read_exact(stream, 8)
    n = struct.unpack('!Q', header)[0]
    if n > 128 * 1024 * 1024:
        raise ValueError('ipc_size_limit')
    body = read_exact(stream, n)
    try:
        return pickle.loads(body)  # Trusted local parent only; not a network endpoint.
    finally:
        wipe(body)


def send_message(stream, message):
    body = bytearray(pickle.dumps(message, protocol=5))
    try:
        stream.write(struct.pack('!Q', len(body)))
        stream.write(body)
        stream.flush()
    finally:
        wipe(body)


def main():
    # stdout is reserved for framed IPC. All third-party prints go to stderr.
    protocol_out = sys.stdout.buffer
    sys.stdout = sys.stderr
    root = Path(os.environ['BEAUTY_ROOT'])
    repo = root / 'HairFastGAN'
    os.chdir(repo)
    sys.path.insert(0, str(repo))
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU required')
    torch.set_num_threads(2)
    service = sys.argv[1]
    try:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        load_started = time.perf_counter()
        engine = HairEngine(root) if service == 'hairstyle' else ColorEngine(root)
        torch.cuda.synchronize()
        startup = {
            'model_load_s': time.perf_counter() - load_started,
            'init_peak_allocated_mib': torch.cuda.max_memory_allocated() / 2**20,
            'init_peak_reserved_mib': torch.cuda.max_memory_reserved() / 2**20,
        }
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        provenance = json.loads((root / 'run_config.json').read_text())
        provenance.update(gpu=torch.cuda.get_device_name(), torch_runtime=torch.__version__,
                          cuda_runtime=torch.version.cuda, seed=3407, engine=service,
                          benchmark_profile='FP32_batch1_no_poisson')
        # Explicit public-sample warmup. Network is used only before user input.
        warmface = bytearray((root / 'public_demo' / '6.png').read_bytes())
        warmref = bytearray((root / 'public_demo' / '7.png').read_bytes())
        sys.addaudithook(audit_guard)
        warmup_started = time.perf_counter()
        for _ in range(2):
            with request_guard():
                evaluate(engine, {'face': warmface.copy(), 'reference': warmref.copy(), 'repeats': 1}, provenance)
        startup['warmup_s'] = time.perf_counter() - warmup_started
        provenance['startup'] = startup
        wipe(warmface)
        wipe(warmref)
        send_message(protocol_out, {'status': 'ready', 'provenance': provenance, 'backend': engine.backend})
    except Exception as exc:
        # Setup has no private images. Its traceback is useful for installation failures.
        import traceback
        traceback.print_exc(file=sys.stderr)
        send_message(protocol_out, {'status': 'init_failed', 'error_code': type(exc).__name__})
        return
    while True:
        request = response = None
        try:
            request = read_message(sys.stdin.buffer)
            if request.get('op') == 'stop':
                send_message(protocol_out, {'status': 'stopped'})
                break
            if request.get('op') != 'evaluate':
                raise ValueError('invalid_operation')
            with request_guard():
                response = evaluate(engine, request, provenance)
            send_message(protocol_out, response)
        except EOFError:
            break
        except Exception as exc:
            # No traceback or original filenames for private requests.
            code = 'cuda_out_of_memory' if isinstance(exc, torch.cuda.OutOfMemoryError) else type(exc).__name__
            known = {'exactly_one_face_required', 'insufficient_hair_pixels', 'insufficient_reference_hair',
                     'invalid_image_bytes', 'image_limits_exceeded', 'unsupported_image_format',
                     'side_effect_blocked_during_request', 'disk_write_blocked_during_request'}
            if str(exc) in known:
                code = str(exc)
            metrics = {'status': 'error', 'error_code': code, 'provenance': provenance}
            if request and request.get('debug_public'):
                import traceback
                metrics['public_diagnostic'] = traceback.format_exc()[-6000:]
            send_message(protocol_out, {'metrics': metrics})
        finally:
            if request is not None:
                wipe(request.get('face'))
                wipe(request.get('reference'))
                request.clear()
            if response is not None:
                wipe(response.get('output_png'))
                wipe(response.get('comparison_png'))
                response.clear()
            engine.clear_private_state()
            gc.collect()
            torch.cuda.empty_cache()


if __name__ == '__main__':
    main()


## سلول 11 — ارزیابی و بنچمارک

ZIP بدون استخراج، ارزیابی انسانی، گزارش بی‌نام و گیت.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/controller.py
# Notebook control plane: stdlib only. Image arrays live in the isolated model worker.
import os, io, sys, json, time, gc, struct, pickle, subprocess, contextlib, zipfile, re, csv
from pathlib import Path, PurePosixPath
from collections import Counter
import select, threading
from collections import deque

PRIVATE_IMAGES = True       # Default: no image display, no return, no image export.
# Set False ONLY for public/synthetic benchmark images. Colab is not an audited private backend.
CLIENT = None
MAX_IPC = 128 * 1024 * 1024

def configure(root, code_root, python, env):
    global ROOT, CODE_ROOT, PYTHON, ENV
    ROOT, CODE_ROOT = Path(root), Path(code_root)
    PYTHON, ENV = str(python), dict(env)


def clear_buffer(buffer):
    if isinstance(buffer, bytearray):
        buffer[:] = b'\0' * len(buffer)
    elif isinstance(buffer, io.BytesIO):
        if buffer.closed:
            return
        view = buffer.getbuffer()
        try:
            view[:] = b'\0' * len(view)
        finally:
            view.release()
            buffer.close()


def recv_exact(stream, size, deadline=None):
    buffer = bytearray(size)
    view = memoryview(buffer)
    offset = 0
    while offset < size:
        if deadline is not None:
            remaining = deadline - time.monotonic()
            if remaining <= 0 or not select.select([stream], [], [], remaining)[0]:
                clear_buffer(buffer)
                raise TimeoutError('model_response_timeout')
        n = stream.readinto(view[offset:])
        if not n:
            clear_buffer(buffer)
            raise RuntimeError('Model worker stopped; restart with start_engine().')
        offset += n
    view.release()
    return buffer


class ModelClient:
    def __init__(self, service):
        if service not in {'hairstyle', 'color'}:
            raise ValueError('Service must be hairstyle or color.')
        self.service = service
        self.initializing = True
        self.init_logs = deque(maxlen=60)
        self.process = subprocess.Popen([PYTHON, str(CODE_ROOT / 'worker.py'), service],
                                        stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                        stderr=subprocess.PIPE, env=ENV, bufsize=0)
        def drain():
            for raw in iter(self.process.stderr.readline, b''):
                if self.initializing:
                    line = raw.decode('utf-8', errors='replace')
                    self.init_logs.append(line)
                    print(line, end='', flush=True)
                # Request-time third-party output is discarded for private inputs.
        self.log_thread = threading.Thread(target=drain, daemon=True)
        self.log_thread.start()
        try:
            ready = self._receive(timeout=1800)
        except Exception as exc:
            self.close()
            raise RuntimeError('Model initialization failed:\n' + ''.join(self.init_logs)
                               + '\nTransport: ' + str(exc)) from exc
        if ready.get('status') != 'ready':
            self.close()
            raise RuntimeError('Model initialization failed:\n' + ''.join(self.init_logs))
        self.initializing = False
        self.ready = ready

    def _receive(self, timeout=180):
        deadline = time.monotonic() + timeout
        header = recv_exact(self.process.stdout, 8, deadline)
        size = struct.unpack('!Q', header)[0]
        if size > MAX_IPC:
            raise RuntimeError('IPC message too large.')
        body = recv_exact(self.process.stdout, size, deadline)
        try:
            return pickle.loads(body)
        finally:
            clear_buffer(body)

    def call(self, request):
        if self.process.poll() is not None:
            raise RuntimeError('Worker not running.')
        body = bytearray(pickle.dumps(request, protocol=5))
        try:
            # FileIO on pipes can short-write. Always write each frame completely.
            for part in (struct.pack('!Q', len(body)), body):
                view = memoryview(part)
                while view:
                    written = self.process.stdin.write(view)
                    if not written:
                        raise RuntimeError('Worker pipe closed.')
                    view = view[written:]
                view.release()
            self.process.stdin.flush()
            try:
                return self._receive()
            except Exception:
                self.close()
                raise
        finally:
            clear_buffer(body)

    def close(self):
        if self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=10)
        if hasattr(self, 'log_thread'):
            self.log_thread.join(timeout=1)
        for pipe in (self.process.stdin, self.process.stdout, self.process.stderr):
            if pipe is not None:
                pipe.close()


def stop_engine():
    global CLIENT
    if CLIENT is not None:
        CLIENT.close()
        CLIENT = None
    gc.collect()


def start_engine(service='hairstyle'):
    """Keep ONE service resident on GPU. Warmup is public, not measured in scores."""
    global CLIENT
    stop_engine()
    CLIENT = ModelClient(service)
    print(json.dumps(CLIENT.ready, indent=2))


async_file_picker_js = r'''
new Promise(resolve => {
  const input = document.createElement('input');
  input.type = 'file'; input.accept = ACCEPT_TOKEN;
  input.style.display = 'block';
  const label = document.createElement('p'); label.textContent = LABEL_TOKEN;
  document.body.append(label, input);
  input.oncancel = () => {input.remove(); label.remove(); resolve(null);};
  input.onchange = async () => {
    const file = input.files[0];
    if (!file || file.size > LIMIT_TOKEN) {
      input.remove(); label.remove(); resolve({error:'invalid_file_size'}); return;
    }
    const reader = new FileReader();
    reader.onload = () => {
      const data = String(reader.result).split(',')[1];
      input.value = ''; input.remove(); label.remove(); resolve({data});
    };
    reader.onerror = () => {input.remove(); label.remove(); resolve({error:'read_failed'});};
    reader.readAsDataURL(file);
  };
})
'''


def pick_bytes(label='Choose image', zip_input=False):
    """FileReader -> memory. Does NOT use google.colab.files.upload (disk write)."""
    import base64
    from google.colab.output import eval_js
    script = async_file_picker_js.replace('ACCEPT_TOKEN', json.dumps('.zip' if zip_input else '.jpg,.jpeg,.png,.webp'))
    script = script.replace('LABEL_TOKEN', json.dumps(label)).replace('LIMIT_TOKEN', str(256 * 2**20 if zip_input else 25 * 2**20))
    response = eval_js(script, timeout_sec=300)
    try:
        if not response or response.get('error'):
            raise ValueError('No valid file selected.')
        return bytearray(base64.b64decode(response['data'], validate=True))
    finally:
        if response is not None:
            response.clear()
        gc.collect()


def show_comparison(png):
    if PRIVATE_IMAGES:
        raise PermissionError('Image display is disabled for private images: notebook outputs can persist.')
    from IPython.display import display, Image as DisplayImage
    display(DisplayImage(data=bytes(png)))


def evaluate(face_bytes, reference_bytes=None, *, show=False, retain_output=False,
             repeats=3, target_rgb=(116, 52, 40), strength=.85, reference_color=False, preview_fn=None):
    """Consumes/zeroes supplied bytearrays. Return only metrics by default.
    Pure inference excludes uploads/decode/render; pipeline_s includes alignment.
    """
    if CLIENT is None:
        raise RuntimeError('Run start_engine() first.')
    if PRIVATE_IMAGES and (show or retain_output):
        raise PermissionError('For private images, display/export/retention is blocked.')
    if not isinstance(face_bytes, bytearray) or (reference_bytes is not None and not isinstance(reference_bytes, bytearray)):
        raise TypeError('Use owned bytearray inputs so they can be overwritten.')
    response = None
    try:
        response = CLIENT.call({'op': 'evaluate', 'face': face_bytes, 'reference': reference_bytes,
                                'repeats': repeats, 'target_rgb': target_rgb, 'strength': strength,
                                'reference_color': reference_color, 'return_images': show or retain_output or preview_fn is not None,
                                'debug_public': not PRIVATE_IMAGES})
        if preview_fn is not None and response['metrics']['status'] == 'ok':
            preview_fn(response['comparison_png'])
        if show and response['metrics']['status'] == 'ok':
            show_comparison(response['comparison_png'])
        if retain_output:
            return response
        return response['metrics']
    finally:
        clear_buffer(face_bytes)
        clear_buffer(reference_bytes)
        if response is not None and not retain_output:
            clear_buffer(response.get('output_png'))
            clear_buffer(response.get('comparison_png'))
        gc.collect()


def single_demo(*, show=False, reference_color=False):
    # No upload data assigned to notebook globals or returned as last-cell output.
    face = reference = None
    try:
        face = pick_bytes('Choose target portrait')
        if CLIENT is None:
            raise RuntimeError('Run start_engine first.')
        if CLIENT.service == 'hairstyle' or reference_color:
            reference = pick_bytes('Choose hair reference (one face)')
        metrics = evaluate(face, reference, show=show, reference_color=reference_color)
        print(json.dumps(metrics, indent=2))
    finally:
        clear_buffer(face)
        clear_buffer(reference)


ID_PATTERN = re.compile(r'[ABC](0[1-9]|10)')


def validate_manifest(cases):
    if not isinstance(cases, list) or not 1 <= len(cases) <= 30:
        raise ValueError('Manifest must contain 1..30 cases.')
    ids = set()
    for case in cases:
        if not isinstance(case, dict) or not ID_PATTERN.fullmatch(str(case.get('id', ''))):
            raise ValueError('Use anonymous IDs A01..A10, B01..B10, C01..C10.')
        if case['id'] in ids or case.get('group') != case['id'][0]:
            raise ValueError('Duplicate ID or inconsistent group.')
        ids.add(case['id'])
        for key in ('target', 'reference'):
            path = case.get(key)
            if key == 'reference' and path is None:
                continue
            if not isinstance(path, str) or not path or '\\' in path:
                raise ValueError('Invalid manifest image path.')
            pure = PurePosixPath(path)
            if pure.is_absolute() or '..' in pure.parts or pure.suffix.lower() not in {'.jpg','.jpeg','.png','.webp'}:
                raise ValueError('Invalid manifest image path.')
    return cases


@contextlib.contextmanager
def zip_cases(zip_bytes):
    """Never extract. Drop compressed image members immediately after last use."""
    stream = io.BytesIO(zip_bytes)
    clear_buffer(zip_bytes)
    try:
        with zipfile.ZipFile(stream) as archive:
            infos = archive.infolist()
            names = [x.filename for x in infos]
            if len(names) > 256 or len(names) != len(set(names)):
                raise ValueError('Invalid ZIP entry count / duplicate entries.')
            if sum(x.file_size for x in infos) > 512 * 2**20:
                raise ValueError('ZIP expansion limit exceeded.')
            if any(x.flag_bits & 1 or x.file_size > 25 * 2**20 or
                   x.file_size / max(1, x.compress_size) > 200 for x in infos):
                raise ValueError('ZIP entry limits exceeded.')
            if 'manifest.json' not in names:
                raise ValueError('ZIP must include manifest.json at its root.')
            info = archive.getinfo('manifest.json')
            if info.file_size > 128 * 1024:
                raise ValueError('Manifest too large.')
            cases = validate_manifest(json.loads(archive.read('manifest.json')))
            needed = [case[key] for case in cases for key in ('target', 'reference') if case.get(key)]
            if any(name not in names for name in needed):
                raise ValueError('Manifest refers to a missing image.')
            remaining = Counter(needed)
            def loader(name):
                return bytearray(archive.read(name))
            def release(name):
                remaining[name] -= 1
                if remaining[name] == 0:
                    entry = archive.getinfo(name)
                    view = stream.getbuffer()
                    start = entry.header_offset
                    header = struct.unpack_from('<4s5H3I2H', view, start)
                    body_start = start + 30 + header[-2] + header[-1]
                    view[body_start:body_start + entry.compress_size] = b'\0' * entry.compress_size
                    view.release()
            yield cases, loader, release
    finally:
        clear_buffer(stream)
        gc.collect()


def record_scores(case, metrics):
    """Human QA after public or transient private preview; never inferred from latency."""
    print('Rate', case['id'], ': 1=unusable, 2=major errors, 3=visible issues, 4=acceptable, 5=excellent')
    while True:
        raw = input('Identity, Hairline, Reference (e.g. 4 4 5): ').split()
        try:
            values = [int(x) for x in raw]
            if len(values) == 3 and all(1 <= x <= 5 for x in values):
                return dict(zip(('identity', 'hairline', 'reference'), values))
        except ValueError:
            pass
        print('Enter three integer scores from 1 to 5.')


def run_batch(cases, loader, *, release=None, show=False, score_fn=None,
              save_dir=None, allow_public_export=False, repeats=3, reference_color=False, preview_fn=None):
    """Sequential batch, no model reload/cache. Sink runs before output-buffer cleanup.
    Private previews use the transient callback; exports stay blocked. Failures count.
    """
    validate_manifest(cases)
    if PRIVATE_IMAGES and (show or save_dir is not None):
        raise PermissionError('Private-image display/export is disabled.')
    if save_dir is not None and not allow_public_export:
        raise PermissionError('Set allow_public_export=True only for public/synthetic data.')
    if CLIENT is None:
        raise RuntimeError('Start an engine before running a batch.')
    records = []
    for case in cases:
        face = reference = response = None
        row = {'id': case['id'], 'group': case['group'], 'engine': CLIENT.service,
               'identity': None, 'hairline': None, 'reference': None}
        try:
            face = loader(case['target'])
            reference = loader(case['reference']) if case.get('reference') else None
            response = evaluate(face, reference, show=show, retain_output=save_dir is not None,
                                repeats=repeats, reference_color=reference_color, preview_fn=preview_fn)
            metrics = response['metrics'] if save_dir is not None else response
            row.update(metrics)
            if metrics['status'] == 'ok':
                if save_dir is not None:
                    dest = Path(save_dir)
                    dest.mkdir(parents=True, exist_ok=True)
                    (dest / (case['id'] + '_result.png')).write_bytes(response['output_png'])
                    (dest / (case['id'] + '_comparison.png')).write_bytes(response['comparison_png'])
                    print('Public comparison ready:', dest / (case['id'] + '_comparison.png'))
                if score_fn is not None:
                    ratings = score_fn(case, metrics)
                    if ratings:
                        for key in ('identity', 'hairline', 'reference'):
                            if isinstance(ratings.get(key), bool) or not isinstance(ratings.get(key), int) or not 1 <= ratings[key] <= 5:
                                raise ValueError('Invalid score.')
                        row.update(ratings)
        except Exception as exc:
            row.update(status='error', error_code=type(exc).__name__)
        finally:
            clear_buffer(face)
            clear_buffer(reference)
            if isinstance(response, dict):
                clear_buffer(response.get('output_png'))
                clear_buffer(response.get('comparison_png'))
            if release:
                for key in ('target', 'reference'):
                    if case.get(key):
                        release(case[key])
        records.append(row)
        print(case['id'], row['status'], 'inference_s=', row.get('inference_s'),
              'peak_MiB=', row.get('peak_allocated_mib'), 'error=', row.get('error_code'))
    return records


def batch_zip_demo(*, show=False, score_fn=None, save_dir=None, allow_public_export=False, preview_fn=None):
    blob = pick_bytes('Choose ZIP with manifest.json and images', zip_input=True)
    try:
        with zip_cases(blob) as (cases, loader, release):
            return run_batch(cases, loader, release=release, show=show, score_fn=score_fn,
                             save_dir=save_dir, allow_public_export=allow_public_export, preview_fn=preview_fn)
    finally:
        clear_buffer(blob)


def batch_folder(folder, *, show=False, score_fn=None, save_dir=None, allow_public_export=False, preview_fn=None):
    """Read a pre-existing folder without modifying input files. PUBLIC DATA ONLY.
    Strict private runs use memory upload instead of creating folders on Colab.
    """
    if PRIVATE_IMAGES:
        raise PermissionError('Folder mode is restricted to pre-existing public/synthetic datasets.')
    root = Path(folder).resolve()
    cases = validate_manifest(json.loads((root / 'manifest.json').read_text()))
    def loader(name):
        path = (root / name).resolve()
        if not path.is_relative_to(root) or not path.is_file() or path.stat().st_size > 25 * 2**20:
            raise ValueError('Invalid image path.')
        return bytearray(path.read_bytes())
    return run_batch(cases, loader, show=show, score_fn=score_fn, save_dir=save_dir,
                     allow_public_export=allow_public_export, preview_fn=preview_fn)


def percentile(values, p):
    ordered = sorted(values)
    if not ordered:
        return None
    k = (len(ordered) - 1) * p / 100
    lo, hi = int(k), min(int(k) + 1, len(ordered) - 1)
    return ordered[lo] * (hi - k) + ordered[hi] * (k - lo) if hi != lo else ordered[lo]


def quality_gate(records, *, threshold=4, seconds=8., latency_key='inference_s'):
    """Provisional preregistered rubric: ALL 3 quality scores >=4.
    Gate: full 30 (10/group), >=21 quality passes AND >=21 joint quality/latency
    passes. P95 is reported separately, not added to the user's gate. Failures count in /30.
    Missing ratings => UNASSESSED. Different engines may not be pooled.
    """
    expected_ids = {f'{g}{i:02d}' for g in 'ABC' for i in range(1, 11)}
    actual = [r.get('id') for r in records]
    engines = {r.get('engine') for r in records}
    full = len(actual) == 30 and set(actual) == expected_ids and len(set(actual)) == 30
    full = full and all(r.get('group') == r['id'][0] for r in records) and len(engines) == 1
    if not 1 <= threshold <= 5 or seconds <= 0:
        raise ValueError('Invalid gate threshold.')
    quality = joint = 0
    missing_latency = 0
    missing = 0
    groups = {g: {'total': 0, 'quality_pass': 0, 'joint_pass': 0, 'errors': 0} for g in 'ABC'}
    times = []
    for row in records:
        group = groups.get(row.get('group'))
        if group is None:
            continue
        group['total'] += 1
        if row.get('status') != 'ok':
            group['errors'] += 1
            continue
        latency = row.get(latency_key)
        if isinstance(latency, (int, float)) and math.isfinite(latency) and latency >= 0:
            times.append(latency)
        else:
            latency = None
            missing_latency += 1
        values = [row.get(k) for k in ('identity', 'hairline', 'reference')]
        valid = all(isinstance(x, int) and not isinstance(x, bool) and 1 <= x <= 5 for x in values)
        if not valid:
            missing += 1
            continue
        good = min(values) >= threshold and not row.get('identity_hard_fail', False)
        fast = latency is not None and latency < seconds
        quality += int(good)
        joint += int(good and fast)
        group['quality_pass'] += int(good)
        group['joint_pass'] += int(good and fast)
    p95 = percentile(times, 95)
    if not full:
        status = 'INCOMPLETE'
    elif missing or missing_latency:
        status = 'UNASSESSED'
    elif quality >= 21 and joint >= 21:
        status = 'PASS'
    else:
        status = 'FAIL'
    return {'gate': status, 'full_protocol': full, 'quality_passes': quality,
            'joint_passes': joint, 'denominator': 30, 'quality_pass_rate': quality / 30,
            'joint_pass_rate': joint / 30, 'unrated_successes': missing,
            'missing_latency': missing_latency,
            'latency_metric': latency_key, 'median_s': percentile(times, 50), 'p95_s': p95,
            'seconds_limit_exclusive': seconds, 'groups': groups,
            'note': 'Scores >=4 are a proposed operational rubric; agree before collecting results.'}


# Non-image reports only: no source paths, original names, photos or latent embeddings.
import math

def export_metrics(records, path='/content/beauty_phase0/metrics.csv', seconds=8.):
    keys = ['id','group','engine','status','error_code','identity','hairline','reference',
            'inference_s','alignment_s','pipeline_s','decode_s','peak_allocated_mib',
            'peak_reserved_mib','nvml_process_peak_mib','nvml_device_peak_mib','repeats']
    with open(path, 'w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(records)
    metadata_path = str(Path(path).with_suffix('.json'))
    with open(metadata_path, 'w') as stream:
        json.dump({'gate': quality_gate(records, seconds=seconds), 'records': records}, stream, indent=2)
    print('Saved anonymous metrics CSV and JSON:', path, metadata_path)


## سلول 12 — دریافت عکس و نمایش موقت

بدنهٔ کامل FileReader و پیش‌نمایش موقت؛ تصاویر در فایل نوشته نمی‌شوند.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/engine/photo_preview.py
"""Colab memory upload and temporary preview; no image files or display_data.
Python/browser/Colab copies cannot be securely erased or independently audited.
"""
import base64
import gc
import json

def pick_image(label):
    from google.colab.output import eval_js
    script = r'''
    new Promise(resolve => {
      const box = document.createElement('section');
      box.dir = 'rtl';
      const title = document.createElement('p'); title.textContent = LABEL;
      const input = document.createElement('input');
      input.type = 'file'; input.accept = '.jpg,.jpeg,.png,.webp';
      const cancel = document.createElement('button'); cancel.textContent = 'لغو';
      box.append(title, input, cancel); document.body.append(box);
      let done = false; let reader = null;
      const finish = value => {
        if (done) return; done = true; clearTimeout(timer);
        if (reader && reader.readyState === 1) reader.abort();
        if (reader) {reader.onload = null; reader.onerror = null; reader = null;}
        input.value = ''; box.remove(); resolve(value);
      };
      const timer = setTimeout(() => finish({error:'timeout'}), 290000);
      cancel.onclick = () => finish({error:'cancelled'});
      input.oncancel = () => finish({error:'cancelled'});
      input.onchange = () => {
        const file = input.files[0];
        if (!file || file.size === 0 || file.size > 25 * 1024 * 1024) {
          finish({error:'size'}); return;
        }
        reader = new FileReader();
        reader.onerror = () => finish({error:'read_failed'});
        reader.onload = () => {
          const data = String(reader.result).split(',')[1];
          finish({data});
        };
        reader.readAsDataURL(file);
      };
    })
    '''.replace('LABEL', json.dumps(label, ensure_ascii=True))
    reply = None
    try:
        reply = eval_js(script, timeout_sec=300)
        if not reply or reply.get('error'):
            messages = {
                'size': 'عکس باید JPEG، PNG یا WEBP و حداکثر ۲۵ مگابایت باشد.',
                'read_failed': 'مرورگر نتوانست عکس را بخواند؛ دوباره همین سلول را اجرا کن.',
                'timeout': 'زمان انتخاب عکس تمام شد؛ دوباره همین سلول را اجرا کن.',
                'cancelled': 'انتخاب عکس لغو شد.',
            }
            raise ValueError(messages.get((reply or {}).get('error'), 'عکسی انتخاب نشد.'))
        return bytearray(base64.b64decode(reply['data'], validate=True))
    finally:
        if reply is not None:
            reply.clear()
        gc.collect()

def show_transient(png):
    from google.colab.output import eval_js
    encoded = script = None
    try:
        encoded = base64.b64encode(png).decode('ascii')
        script = r'''
        (() => {
          if (window.__beautyDisposePreview) window.__beautyDisposePreview();
          const box = document.createElement('section'); box.dir = 'rtl';
          const title = document.createElement('p');
          title.textContent = 'از چپ: عکس اصلی، چهرهٔ هم‌تراز، مرجع مو، نتیجه';
          const image = document.createElement('img');
          image.style.cssText = 'display:block;max-width:100%;height:auto';
          image.alt = 'مقایسهٔ عکس ورودی و نتیجهٔ انتقال مو';
          const close = document.createElement('button');
          close.textContent = 'بستن و پاک‌کردن پیش‌نمایش';
          let text = atob(PAYLOAD);
          const pixels = Uint8Array.from(text, c => c.charCodeAt(0)); text = '';
          let url = URL.createObjectURL(new Blob([pixels], {type:'image/png'}));
          pixels.fill(0);
          image.src = url; box.append(title, image, close); document.body.append(box);
          let timer;
          const dispose = () => {
            clearTimeout(timer); image.removeAttribute('src');
            if (url) URL.revokeObjectURL(url); url = null; box.remove();
            window.removeEventListener('pagehide', dispose);
            if (window.__beautyDisposePreview === dispose) delete window.__beautyDisposePreview;
          };
          window.__beautyDisposePreview = dispose;
          close.onclick = dispose;
          window.addEventListener('pagehide', dispose, {once:true});
          timer = setTimeout(dispose, 300000);
          return new Promise(resolve => {
            image.onload = () => {image.onload = null; image.onerror = null; resolve(true);};
            image.onerror = () => {dispose(); resolve(false);};
          });
        })()
        '''.replace('PAYLOAD', json.dumps(encoded))
        if eval_js(script, timeout_sec=30) is not True:
            raise RuntimeError('نمایش تصویر در مرورگر موفق نشد؛ دوباره همین سلول را اجرا کن.')
    finally:
        encoded = script = None
        gc.collect()

def run(service='hairstyle', repeats=1, reference_color=False,
        target_rgb=(116, 52, 40), strength=.85):
    """One interactive case; return anonymous metrics, consume all image buffers."""
    if service not in {'hairstyle', 'color'}:
        raise ValueError('Unknown service')
    import runtime
    api = runtime.api()
    face = reference = response = None
    try:
        # Stop another service before loading HairFastGAN, so T4 holds one engine.
        print('۱/۴ — آماده‌سازی موتور ' + service + '؛ این زمان جزو inference نیست.', flush=True)
        if (api.CLIENT is None or api.CLIENT.service != service
                or api.CLIENT.process.poll() is not None):
            api.start_engine(service)
        print('۲/۴ — عکس فرد را از دستگاه خودت انتخاب کن.', flush=True)
        face = pick_image('عکس فرد: یک چهرهٔ واضح، ترجیحاً روبه‌رو')
        if service == 'hairstyle' or reference_color:
            print('۳/۴ — عکس مرجع مو را انتخاب کن.', flush=True)
            reference = pick_image('عکس مرجع مو: فقط یک چهره، موها واضح و قابل مشاهده')
        print('۴/۴ — پردازش عکس‌های انتخاب‌شده…', flush=True)
        # This explicit in-memory path allows ephemeral preview while preserving
        # PRIVATE_IMAGES and suppressing request-time private diagnostics.
        response = api.CLIENT.call({
            'op': 'evaluate', 'face': face, 'reference': reference,
            'repeats': repeats, 'return_images': True, 'debug_public': False,
            'reference_color': reference_color, 'target_rgb': target_rgb, 'strength': strength,
        })
        metrics = response.get('metrics', {})
        if metrics.get('status') != 'ok':
            error = metrics.get('error_code', 'unknown_error')
            explanations = {
                'exactly_one_face_required': 'در هر عکس باید دقیقاً یک چهرهٔ قابل تشخیص باشد.',
                'cuda_out_of_memory': 'حافظهٔ GPU کافی نبود؛ پردازش GPU دیگری را متوقف کن.',
                'image_limits_exceeded': 'عکس باید ثابت و حداکثر ۱۶ میلیون پیکسل باشد.',
                'unsupported_image_format': 'فقط عکس JPEG، PNG یا WEBP ثابت پشتیبانی می‌شود.',
                'invalid_image_bytes': 'فایل عکس معتبر نیست یا از محدودیت حجم عبور کرده است.',
            }
            raise RuntimeError(explanations.get(error, 'پردازش موفق نشد.') + ' [' + str(error) + ']')
        print('زمان خالص: %.3f ثانیه | پایپ‌لاین: %.3f ثانیه | Peak allocated: %.1f MiB'
              % (metrics['inference_s'], metrics['pipeline_s'], metrics['peak_allocated_mib']), flush=True)
        # Erase parent inputs and the separate result before rendering comparison.
        api.clear_buffer(face); api.clear_buffer(reference)
        api.clear_buffer(response.get('output_png'))
        show_transient(response['comparison_png'])
        print('پیش‌نمایش آماده است؛ پس از ۵ دقیقه بسته می‌شود. این تست تک‌عکس است، نه گیت ۳۰ عکس.', flush=True)
        return dict(metrics)
    except (ValueError, RuntimeError) as exc:
        print('متوقف شد: ' + str(exc), flush=True)
        return {'status': 'error', 'error_code': type(exc).__name__}
    finally:
        api.clear_buffer(face); api.clear_buffer(reference)
        if response is not None:
            api.clear_buffer(response.get('output_png'))
            api.clear_buffer(response.get('comparison_png'))
            response.clear()
        face = reference = response = None
        api.stop_engine()
        gc.collect()


## سلول 13 — تست‌های تصویر و گیت

JPEG/PNG/WEBP، متادیتا، ماسک، ZIP، محدودیت‌ها و گیت.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/tests/test_core.py
import io, json, os, sys, ast, unittest, zipfile, copy
from pathlib import Path
import numpy as np
from PIL import Image
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'engine'))
import core
import controller as c


def png(color=(80, 90, 100), size=(32, 32)):
    im = Image.new('RGB', size, color)
    stream = io.BytesIO(); im.save(stream, format='PNG')
    data = bytearray(stream.getvalue()); core.wipe(im); core.wipe(stream)
    return data


def good_records():
    return [{'id':f'{g}{i:02d}', 'group':g, 'engine':'hairstyle', 'status':'ok',
             'identity':4,'hairline':4,'reference':4,'pipeline_s':7.,'inference_s':5.}
            for g in 'ABC' for i in range(1,11)]


class Tests(unittest.TestCase):
    def test_decode_jpeg_png_webp_cleanup_and_pixels(self):
        for fmt in ('JPEG', 'PNG', 'WEBP'):
            with self.subTest(format=fmt):
                image=Image.new('RGB',(40,30),(80,90,100))
                stream=io.BytesIO()
                image.save(stream,format=fmt)
                blob=bytearray(stream.getvalue())
                clean=core.decode_clean(blob)
                try:
                    self.assertEqual(clean.size,(40,30))
                    self.assertFalse(clean.info)
                    self.assertFalse(clean.getexif())
                    self.assertLessEqual(max(abs(v-w) for v,w in zip(clean.getpixel((20,15)),(80,90,100))),2)
                finally:
                    core.wipe(clean);core.wipe(clean)
                    core.wipe(blob);stream.close();image.close()

    def test_encoded_buffer_wiped_before_external_close(self):
        stream=core.WipingBytesIO(b'private encoded bytes')
        # A caller-held view permits inspection after zeroing and before release.
        view=io.BytesIO.getbuffer(stream)
        try:
            with self.assertRaises(BufferError):
                stream.close()
            self.assertEqual(bytes(view),b'\0'*len(view))
        finally:
            view.release()
            stream.close()
        core.wipe(stream);core.wipe(stream)
        closed=io.BytesIO(b'fixture');closed.close()
        core.wipe(closed);c.clear_buffer(closed)

    def test_official_public_demo_pngs_and_comparison_roundtrip(self):
        root=Path(os.environ.get('BEAUTY_TEST_DEMO','/content/beauty_phase0/public_demo'))
        if not all((root/name).is_file() for name in ('6.png','7.png')):
            self.skipTest('Official public demo assets have not been downloaded.')
        for name in ('6.png','7.png'):
            with self.subTest(public_demo=name):
                blob=bytearray((root/name).read_bytes())
                image=frame=decoded=None
                encoded=None
                try:
                    image=core.decode_clean(blob)
                    self.assertGreater(min(image.size),100)
                    self.assertFalse(image.info)
                    frame=core.comparison_frame(image,image,image,image)
                    encoded=core.encode_png(frame)
                    decoded=core.decode_clean(encoded)
                    self.assertEqual(decoded.size,frame.size)
                finally:
                    for value in (image,frame,decoded,encoded,blob):
                        if value is not None:core.wipe(value)

    def test_rejected_image_does_not_decode_during_cleanup(self):
        from unittest.mock import patch
        source=png(size=(32,32))
        image=Image.open(io.BytesIO(source))
        old=core.MAX_PIXELS;core.MAX_PIXELS=10
        try:
            with patch.object(core.Image,'open',return_value=image),patch.object(image,'load',side_effect=AssertionError('rejected image decoded')):
                with self.assertRaisesRegex(ValueError,'image_limits_exceeded'):
                    core.decode_clean(source)
        finally:
            core.MAX_PIXELS=old;core.wipe(source);image.close()

    def test_metadata_and_orientation(self):
        im = Image.new('RGB',(40,20),(50,100,150))
        exif = Image.Exif(); exif[274]=6; exif[270]='PRIVATE_METADATA'
        stream=io.BytesIO(); im.save(stream,format='JPEG',exif=exif)
        clean=core.decode_clean(bytearray(stream.getvalue()))
        self.assertEqual(clean.size,(20,40));self.assertFalse(clean.info);self.assertFalse(clean.getexif())
        result=core.encode_png(clean)
        self.assertNotIn(b'PRIVATE_METADATA',result)
        self.assertNotIn(b'eXIf',result)
        core.wipe(clean);core.wipe(result)

    def test_rgb_limits(self):
        with self.assertRaises(ValueError):core.decode_clean(bytearray())
        with self.assertRaises(Exception):core.decode_clean(bytearray(b'not an image'))
        old=core.MAX_PIXELS;core.MAX_PIXELS=10
        try:
            with self.assertRaises(ValueError):core.decode_clean(png())
        finally:core.MAX_PIXELS=old

    def test_mask_and_zero_strength(self):
        rgb=np.zeros((32,32,3),np.uint8);rgb[:]=[80,65,55]
        mask=np.zeros((32,32),np.float32);mask[8:24,8:24]=1
        out=core.lab_recolor(rgb,mask,(180,35,20),1.)
        np.testing.assert_array_equal(out[mask==0],rgb[mask==0])
        self.assertGreater(np.abs(out[mask>0].astype(float)-rgb[mask>0]).mean(),5.)
        unchanged=core.lab_recolor(rgb,mask,(180,35,20),0.)
        self.assertLessEqual(np.abs(unchanged.astype(int)-rgb.astype(int)).max(),1)
        with self.assertRaises(ValueError):core.lab_recolor(rgb,np.zeros_like(mask),(1,2,3))
        with self.assertRaises(ValueError):core.lab_recolor(rgb,mask,(999,2,3))

    def test_luminance_preserved_for_in_gamut_target(self):
        import cv2
        rgb=np.empty((32,32,3),np.uint8)
        for y in range(32):rgb[y]=[100+y,90+y,80+y]
        out=core.lab_recolor(rgb,np.ones((32,32),np.float32),(130,100,100),.5)
        before=cv2.cvtColor(rgb.astype(np.float32)/255,cv2.COLOR_RGB2LAB)[...,0]
        after=cv2.cvtColor(out.astype(np.float32)/255,cv2.COLOR_RGB2LAB)[...,0]
        self.assertLess(np.abs(before-after).max(),.5)

    def test_gate_21_and_failures_in_denominator(self):
        rows=good_records()
        for r in rows[21:]:r['status']='error'
        g=c.quality_gate(rows)
        self.assertEqual(g['gate'],'PASS');self.assertEqual(g['joint_passes'],21)
        self.assertEqual(g['denominator'],30)
        rows[20]['status']='error';self.assertEqual(c.quality_gate(rows)['gate'],'FAIL')

    def test_gate_missing_duplicate_and_mixed(self):
        rows=good_records();rows[0]['identity']=None
        self.assertEqual(c.quality_gate(rows)['gate'],'UNASSESSED')
        rows=good_records();rows[0]['id']=rows[1]['id']
        self.assertEqual(c.quality_gate(rows)['gate'],'INCOMPLETE')
        rows=good_records();rows[0]['engine']='color'
        self.assertEqual(c.quality_gate(rows)['gate'],'INCOMPLETE')
        self.assertEqual(c.quality_gate(good_records()[:21])['gate'],'INCOMPLETE')

    def test_gate_latency_boundary_and_tail(self):
        rows=good_records()
        for row in rows:row['inference_s']=8.
        self.assertEqual(c.quality_gate(rows)['gate'],'FAIL')
        rows=good_records();rows[0]['inference_s']=40.;rows[1]['inference_s']=40.
        self.assertEqual(c.quality_gate(rows)['gate'],'PASS')
        rows=good_records();rows[0]['inference_s']=float('nan')
        self.assertNotEqual(c.quality_gate(rows)['gate'],'PASS')

    def test_private_guards(self):
        c.PRIVATE_IMAGES=True
        with self.assertRaises(PermissionError):c.show_comparison(bytearray())
        with self.assertRaises(PermissionError):c.batch_folder('/irrelevant')
        case={'id':'A01','group':'A','target':'x.png','reference':'r.png'}
        with self.assertRaises(PermissionError):c.run_batch([case],None,save_dir='/irrelevant')

    def test_manifest_validation(self):
        case={'id':'A01','group':'A','target':'x.png','reference':'r.png'}
        c.validate_manifest([case])
        for path in ['/absolute.png','../secret.png','x/../../secret.png','x.exe','x\\y.png']:
            bad=dict(case,target=path)
            with self.assertRaises(ValueError):c.validate_manifest([bad])
        with self.assertRaises(ValueError):c.validate_manifest([case,case])

    def test_zip_no_extract_and_zero_after_last_use(self):
        case={'id':'A01','group':'A','target':'x.png','reference':'r.png'}
        stream=io.BytesIO()
        with zipfile.ZipFile(stream,'w',zipfile.ZIP_DEFLATED) as archive:
            archive.writestr('manifest.json',json.dumps([case]));archive.writestr('x.png',png());archive.writestr('r.png',png((10,20,30)))
        blob=bytearray(stream.getvalue())
        with c.zip_cases(blob) as (cases,loader,release):
            self.assertEqual(set(blob),{0})
            b=loader('x.png');self.assertTrue(b.startswith(b'\x89PNG'))
            release('x.png')
            with self.assertRaises(Exception):loader('x.png')
            self.assertTrue(loader('r.png').startswith(b'\x89PNG'));release('r.png')
        core.wipe(b)

    def test_batch_failure_consumes_buffers(self):
        class Dummy:
            service='hairstyle'
            def call(self,req):raise RuntimeError('simulated')
        old=c.CLIENT;c.CLIENT=Dummy();c.PRIVATE_IMAGES=True
        buffers=[]
        def loader(name):
            b=png();buffers.append(b);return b
        case={'id':'A01','group':'A','target':'x.png','reference':'r.png'}
        try:
            rows=c.run_batch([case],loader)
            self.assertEqual(rows[0]['status'],'error')
            for b in buffers:self.assertEqual(set(b),{0})
        finally:c.CLIENT=old

    def test_audit_guard_disk_network_inference(self):
        ns={'__name__':'not_main'}
        exec((Path(__file__).resolve().parents[1]/'engine/worker_tail.py').read_text(),ns)
        ns['GUARD_ACTIVE']=True
        for event,args in [('open',('/private','wb',0)),('open',('/private',None,os.O_WRONLY)),
                           ('subprocess.Popen',('x',)),('socket.connect',('x',))]:
            with self.assertRaises(PermissionError):ns['audit_guard'](event,args)
        ns['audit_guard']('open',('/public','rb',0))


if __name__=='__main__':
    unittest.main(verbosity=2)


## سلول 14 — تست ارتباط و Runtime

Timeout، خطای نصب، پاکسازی و جلوگیری از نتیجهٔ قدیمی.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/tests/test_runtime.py
"""CPU regression tests for notebook state, diagnostics and framed IPC; no model inference."""
import contextlib, io, json, os, pathlib, sys, tempfile, time, unittest
sys.path.insert(0,str(pathlib.Path(__file__).resolve().parents[1] / 'engine'))
import controller as api
import runtime as runtime

FAKE_WORKER = r'''
import sys, struct, pickle
def send(value):
    body=pickle.dumps(value,protocol=5)
    sys.stdout.buffer.write(struct.pack('!Q',len(body))+body)
    sys.stdout.buffer.flush()
send({'status':'ready'})
while True:
    header=sys.stdin.buffer.read(8)
    if not header: break
    length=struct.unpack('!Q',header)[0]
    request=pickle.loads(sys.stdin.buffer.read(length))
    send({'metrics':{'status':'ok','fixture':True}})
'''

class RuntimeTests(unittest.TestCase):
    def setUp(self):
        self.temp=tempfile.TemporaryDirectory()
        self.root=pathlib.Path(self.temp.name)
        self.saved=runtime.CODE_ROOT
        runtime.CODE_ROOT=self.root
        api.configure(self.root,self.root,sys.executable,dict(os.environ))

    def tearDown(self):
        api.stop_engine()
        runtime.CODE_ROOT=self.saved
        self.temp.cleanup()

    def test_stage_failure_has_type_without_private_message(self):
        with self.assertRaises(ValueError),contextlib.redirect_stdout(io.StringIO()):
            with runtime.stage(8,'fixture'):
                raise ValueError('PRIVATE_INPUT_FILENAME')
        raw=(self.root/'run_status.json').read_text()
        self.assertNotIn('PRIVATE_INPUT_FILENAME',raw)
        self.assertEqual(json.loads(raw)['8']['status'],'FAILED')

    def test_stage_success_persisted(self):
        with contextlib.redirect_stdout(io.StringIO()):
            with runtime.stage(4,'fixture'): pass
        self.assertEqual(json.loads((self.root/'run_status.json').read_text())['4']['status'],'PASSED')

    def test_pipe_deadline(self):
        readfd,writefd=os.pipe()
        try:
            with os.fdopen(readfd,'rb',buffering=0) as stream:
                with self.assertRaises(TimeoutError):
                    api.recv_exact(stream,8,time.monotonic()+.03)
        finally:
            os.close(writefd)

    def test_transport_consumes_inputs_and_closes_worker(self):
        (self.root/'worker.py').write_text(FAKE_WORKER)
        client=api.ModelClient('hairstyle')
        api.CLIENT=client
        face,reference=bytearray(b'fixture_face'),bytearray(b'fixture_ref')
        result=api.evaluate(face,reference)
        self.assertTrue(result['fixture'])
        self.assertEqual(set(face),{0})
        self.assertEqual(set(reference),{0})
        api.stop_engine()
        self.assertIsNotNone(client.process.poll())

    def test_initialization_failure_contains_original_diagnostic(self):
        fake=FAKE_WORKER.split("send({'status':'ready'})")[0]
        fake+="print('fixture backend error',file=sys.stderr,flush=True)\nsend({'status':'init_failed'})\n"
        (self.root/'worker.py').write_text(fake)
        with contextlib.redirect_stdout(io.StringIO()):
            with self.assertRaisesRegex(RuntimeError,'fixture backend error'):
                api.ModelClient('hairstyle')

    def test_installer_failure_contains_actual_output(self):
        with contextlib.redirect_stdout(io.StringIO()):
            with self.assertRaisesRegex(RuntimeError,'fixture_install_failure'):
                runtime.run(sys.executable,'-c',"print('fixture_install_failure');raise SystemExit(2)")

    def test_initialization_crash_contains_original_diagnostic(self):
        (self.root/'worker.py').write_text("import sys\nprint('fixture fatal import',file=sys.stderr,flush=True)\nraise SystemExit(3)\n")
        with contextlib.redirect_stdout(io.StringIO()):
            with self.assertRaisesRegex(RuntimeError,'fixture fatal import'):
                api.ModelClient('hairstyle')

    def test_summary_rejects_stale_metrics(self):
        import types
        from unittest.mock import patch, Mock
        display_module=types.ModuleType('IPython.display')
        display_module.display=Mock()
        display_module.Markdown=lambda value:value
        (self.root/'hairstyle_demo_metrics.json').write_text(json.dumps({
            'status':'ok','inference_s':1,'peak_allocated_mib':2}))
        with patch.dict(sys.modules,{'IPython.display':display_module}),contextlib.redirect_stdout(io.StringIO()):
            with self.assertRaisesRegex(RuntimeError,'17'):
                runtime.summary()

    def test_fresh_kernel_module_configuration(self):
        import importlib.util
        path=pathlib.Path(__file__).resolve().parents[1]/'engine/runtime.py'
        first=importlib.util.spec_from_file_location('fixture_runtime_one',path)
        second=importlib.util.spec_from_file_location('fixture_runtime_two',path)
        a,b=importlib.util.module_from_spec(first),importlib.util.module_from_spec(second)
        first.loader.exec_module(a);second.loader.exec_module(b)
        self.assertEqual(a.ROOT,b.ROOT)
        self.assertEqual(a.PYTHON,b.PYTHON)
        self.assertEqual(a.ENV['HF_HUB_OFFLINE'],'1')

if __name__=='__main__': unittest.main(verbosity=2)


## سلول 15 — تست شش اپراتور

در سلامت GPU با assets واقعی اجرا می‌شود؛ CI بدون مدل آن را skip می‌کند.


In [ ]:
%%writefile /content/beauty_phase0/benchmark_code/tests/test_operators.py
import importlib.util, os, pathlib, unittest
os.environ['BEAUTY_OP_BACKEND']='torch_reference'
torch = None
if os.environ.get('BEAUTY_TEST_REPO'):
    import torch

class Operators(unittest.TestCase):
    @unittest.skipUnless(os.environ.get('BEAUTY_TEST_REPO'), 'GPU setup assets are not configured; CPU CI does not download models.')
    def test_all_six_pinned_operator_copies(self):
        count=0
        for path in pathlib.Path(os.environ.get('BEAUTY_TEST_REPO', 'research/repo_patched')).rglob('op/*.py'):
            if path.name not in {'fused_act.py','upfirdn2d.py'}:continue
            spec=importlib.util.spec_from_file_location('operator_'+str(count),path)
            module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
            x=torch.arange(-8.,8.).reshape(1,1,4,4)
            if path.name=='fused_act.py':
                y=module.fused_leaky_relu(x,torch.tensor([1.]),negative_slope=.1,scale=2.)
                expected=torch.nn.functional.leaky_relu(x+1.,negative_slope=.1)*2.
                torch.testing.assert_close(y,expected)
                layer=module.FusedLeakyReLU(1)
                self.assertIn('bias',layer.state_dict())
                self.assertIsNone(module.fused)
            else:
                y=module.upfirdn2d(x,torch.ones(1,1),up=2)
                self.assertEqual(y.shape,(1,1,8,8))
                torch.testing.assert_close(y[:,:,::2,::2],x)
                self.assertEqual(float(y[:,:,1::2,:].abs().sum()),0.)
                z=module.upfirdn2d(y,torch.ones(1,1),down=2)
                torch.testing.assert_close(z,x)
                self.assertIsNone(module.upfirdn2d_op)
            count+=1
        self.assertEqual(count,6)

if __name__=='__main__':unittest.main(verbosity=2)


## سلول 16 — ساخت worker و اجرای سلامت

محاسبهٔ CUDA و ۲۶ تست اولیه؛ این مرحله جای استنتاج مدل را نمی‌گیرد.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

import ast
worker = (ctx.CODE_ROOT/'core.py').read_text()+'\n\n'+(ctx.CODE_ROOT/'worker_tail.py').read_text()
ast.parse(worker)
(ctx.CODE_ROOT/'worker.py').write_text(worker)
with ctx.stage(16, 'سلامت CUDA و تست‌ها'):
    ctx.verify()


## سلول 17 — دموی عمومی HairFastGAN

دو warmup و سه تکرار زمان‌سنجی؛ نتیجهٔ crop هم‌تراز ۱۰۲۴×۱۰۲۴ است.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

with ctx.stage(17, 'دموی HairFastGAN'):
    ctx.demo('hairstyle')


## سلول 18 — دموی عمومی رنگ مو

موتور مستقل SegFormer + LAB؛ زمان‌سنجی با انتقال مدل مو ترکیب نمی‌شود.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

with ctx.stage(18, 'دموی رنگ مو'):
    ctx.demo('color')


## سلول 19 — گزارش همین اجرا

فقط دموهای موفق همین اجرا گزارش می‌شوند؛ گیت ۳۰ عکس هنوز جداست.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

with ctx.stage(19, 'گزارش دمو'):
    ctx.summary()


## سلول 20 — عکس فرد + مرجع مدل مو

روشن است. ابتدا عکس فرد و سپس مرجع را انتخاب کن. برای عکس تازه فقط همین سلول را دوباره اجرا کن؛ نصب تکرار نمی‌شود.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

RUN_SINGLE = True
if RUN_SINGLE:
    import photo_preview
    single_metrics = photo_preview.run(service='hairstyle', repeats=1)
else:
    print('تست تک‌عکس غیرفعال است.')


## سلول 21 — تست اختیاری رنگ از مرجع

برای تست، RUN_COLOR را True کن؛ فرم مو تغییر نمی‌کند و رنگ مرجع استخراج می‌شود.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

RUN_COLOR = False
if RUN_COLOR:
    import photo_preview
    color_metrics = photo_preview.run(service='color', repeats=1, reference_color=True)
else:
    print('تست رنگ از مرجع غیرفعال است.')


## سلول 22 — بنچمارک ۳۰ عکس

برای اجرا RUN_BATCH را True کن. ZIP با manifest.json و عکس‌ها در حافظه خوانده می‌شود. هر نتیجه موقت نمایش داده شده و سه نمرهٔ انسانی دریافت می‌شود. فایل عکس ذخیره نمی‌شود؛ CSV/JSON بی‌نام ذخیره می‌شوند.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

RUN_BATCH = False
SERVICE = 'hairstyle'
if RUN_BATCH:
    with ctx.stage(22, 'بنچمارک ۳۰ عکس'):
        batch_gate = ctx.batch(service=SERVICE, public=False, review=True)
else:
    print('بنچمارک ۳۰ عکس هنوز اجرا نشده است.')


## سلول 23 — آزادسازی منابع

پردازش مدل بسته می‌شود. تصویر موقت مرورگر تا بستن یا پایان مهلت قابل مشاهده است.


In [ ]:
import os, pathlib, sys
_code = pathlib.Path('/content/beauty_phase0/benchmark_code/engine')
if not (_code/'runtime.py').is_file():
    raise RuntimeError('ابتدا سلول‌های ۱ و ۲ را اجرا کن؛ کد پایه در این Runtime موجود نیست.')
os.environ['BEAUTY_ROOT'] = '/content/beauty_phase0'
os.environ['BEAUTY_TEST_ROOT'] = str(_code.parent/'tests')
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import runtime as ctx

ctx.stop()
print('پردازش مدل بسته شد؛ برای عکس جدید فقط سلول ۲۰ را اجرا کن.')


## معیارها و محدوده

گروه‌های A/B/C هرکدام ۱۰ عکس؛ شناسه‌های A01..A10، B01..B10، C01..C10. گیت HairFastGAN: هر سه نمرهٔ Identity/Hairline/Reference حداقل ۴، حداقل ۲۱ خروجی با کیفیت قابل قبول و حداقل ۲۱ خروجی که همان کیفیت و `inference_s < 8` را همزمان دارند. آستانهٔ ۴ تعریف عملیاتی پیشنهادی است. خطاها در مخرج ۳۰ باقی می‌مانند؛ دیتاست ناقص INCOMPLETE و نمره/زمان ناموجود UNASSESSED است. P95 فقط گزارش می‌شود. موتور رنگ با حد مستقل ۰٫۵ ثانیه ارزیابی می‌شود و نتایج دو موتور ادغام نمی‌شوند.

`inference_s` پردازش خالص، `alignment_s` هم‌ترازی، `pipeline_s` آماده‌سازی لازم تا تولید تصویر، `decode_s` خواندن/پاکسازی ورودی است. بارگذاری مدل و warmup خارج از زمان inference هستند. Peak allocated/reserved از CUDA و process/device از NVML با نمونه‌برداری ۱۰ ms گزارش می‌شوند؛ NVML ممکن است جهش کوتاه‌تر را نبیند.

StableHair v2، PSGAN/SCGAN و آرایش FaceMesh در این نسخه اجرا نشده‌اند. موبایل، UI محصول و دیتابیس تولید نشده‌اند.

منابع: https://github.com/AIRI-Institute/HairFastGAN ؛ https://huggingface.co/AIRI-Institute/HairFastGAN ؛ https://huggingface.co/isjackwild/segformer-b0-finetuned-segments-skin-hair-clothing ؛ https://github.com/openai/CLIP ؛ https://github.com/ZPdesu/SEAN
